# Pipelines dalam Machine Learning

**Pipeline** adalah cara menggabungkan langkah-langkah preprocessing dan pemodelan menjadi satu objek tunggal yang rapi. Dengan pipeline, proses fit dan transform dilakukan secara otomatis dan berurutan tanpa perlu memanggil setiap langkah secara manual.

---

## Mengapa Menggunakan Pipeline?

| Tanpa Pipeline | Dengan Pipeline |
|---|---|
| Preprocessing dan model dijalankan terpisah | Semua langkah terbungkus dalam satu objek |
| Rawan kebocoran data (*data leakage*) jika tidak hati-hati | Mencegah kebocoran data secara otomatis |
| Kode lebih panjang dan repetitif | Kode lebih bersih dan mudah dibaca |
| Sulit di-deploy | Mudah di-deploy sebagai satu objek |

---

## Alur Kerja Pipeline

```
Data Mentah
    │
    ▼
Pemisahan Kolom (numerik & kategorik)
    │
    ▼
ColumnTransformer (preprocessing paralel)
    ├── Kolom Numerik   → SimpleImputer
    └── Kolom Kategorik → SimpleImputer → OneHotEncoder
    │
    ▼
Model (RandomForestRegressor)
    │
    ▼
Prediksi
```

---

## Import Library

In [1]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import pandas as pd
from sklearn.model_selection import train_test_split

## Load Data

Baca dataset training (`train.csv`) dan test (`test.csv`). Parameter `index_col='Id'` menjadikan kolom `Id` sebagai index baris.

In [2]:
X_1 = pd.read_csv("D:/file baru/mendata/Intermediate Machine Learning/train.csv", index_col='Id')
X_2 = pd.read_csv("D:/file baru/mendata/Intermediate Machine Learning/test.csv", index_col='Id')

## Persiapan Data & Pemilihan Kolom

### Langkah-langkah:
1. Hapus baris yang tidak memiliki label target (`SalePrice`)
2. Pisahkan target `y` dari fitur `X`
3. Split data menjadi **training set (80%)** dan **validation set (20%)**
4. Pilih kolom yang akan digunakan:
   - **Kategorik**: tipe `object` dengan jumlah nilai unik < 10 (menghindari *high cardinality*)
   - **Numerik**: tipe `int64` atau `float64`

> **Catatan:** Kolom kategorik dengan nilai unik terlalu banyak dikeluarkan karena One-Hot Encoding pada kolom tersebut akan menghasilkan terlalu banyak fitur baru.

In [3]:
X_1.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_1.SalePrice
X_1.drop(['SalePrice'], axis=1, inplace=True)
# Break off validation set from training data
X_train, X_valid, y_train, y_valid = train_test_split(X_1, y, 
                                                                train_size=0.8, test_size=0.2,
                                                                random_state=0)
categorical_cols = [cname for cname in X_train.columns if
                    X_train[cname].nunique() < 10 and 
                    X_train[cname].dtype == "object"]

# Select numerical columns
numerical_cols = [cname for cname in X_train.columns if 
                X_train[cname].dtype in ['int64', 'float64']]

# Keep selected columns only
my_cols = categorical_cols + numerical_cols
X_train = X_train[my_cols].copy()
X_valid = X_valid[my_cols].copy()
X_test = X_2[my_cols].copy()

## Preview Data

Tampilkan 10 baris pertama dataset untuk memastikan data berhasil dimuat dan kolom sudah benar.
Perhatikan ada beberapa sel bernilai `NaN` — inilah yang akan ditangani oleh `SimpleImputer` di dalam pipeline.

In [4]:
print(X_1.head(10))

    MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
Id                                                                    
1           60       RL         65.0     8450   Pave   NaN      Reg   
2           20       RL         80.0     9600   Pave   NaN      Reg   
3           60       RL         68.0    11250   Pave   NaN      IR1   
4           70       RL         60.0     9550   Pave   NaN      IR1   
5           60       RL         84.0    14260   Pave   NaN      IR1   
6           50       RL         85.0    14115   Pave   NaN      IR1   
7           20       RL         75.0    10084   Pave   NaN      Reg   
8           60       RL          NaN    10382   Pave   NaN      IR1   
9           50       RM         51.0     6120   Pave   NaN      Reg   
10         190       RL         50.0     7420   Pave   NaN      Reg   

   LandContour Utilities LotConfig  ... ScreenPorch PoolArea PoolQC  Fence  \
Id                                  ...                              

## Membangun Pipeline

### Langkah 1 — Preprocessing Numerik

```python
numerical_transformer = SimpleImputer(strategy='constant')
```
`strategy='constant'` mengisi missing value dengan `0` (nilai default fill_value). Pilihan lain: `'mean'`, `'median'`, `'most_frequent'`.

---

### Langkah 2 — Preprocessing Kategorik

Dua langkah dijalankan **berurutan** di dalam sub-pipeline:
1. **`SimpleImputer(strategy='most_frequent')`** → mengisi missing value dengan nilai yang paling sering muncul di kolom tersebut
2. **`OneHotEncoder(handle_unknown='ignore')`** → mengubah teks menjadi kolom biner (0/1) per kategori
   - `handle_unknown='ignore'` → kategori baru yang muncul di data test tidak menyebabkan error

---

### Langkah 3 — `ColumnTransformer`

`ColumnTransformer` menerapkan transformer yang **berbeda** ke kolom yang **berbeda** secara paralel, lalu menggabungkan hasilnya:

| Parameter | Penjelasan |
|---|---|
| `'num'` / `'cat'` | Nama transformer (bebas) |
| `numerical_transformer` | Objek transformer yang digunakan |
| `numerical_cols` | Daftar kolom yang ditransformasi |

---

### Langkah 4 — Pipeline Final

`Pipeline` final menyatukan `preprocessor` dan `model` menjadi satu objek `clf`:
- Saat `clf.fit()` dipanggil → preprocessing **dipelajari** dari `X_train`, lalu model dilatih
- Saat `clf.predict()` dipanggil → preprocessing **diterapkan** ke data baru, lalu model memprediksi

> Tidak perlu memanggil `transform` secara manual — pipeline menanganinya otomatis.

In [5]:
# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='constant')

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Define model
model = RandomForestRegressor(n_estimators=100, random_state=0)

# Bundle preprocessing and modeling code in a pipeline
clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model', model)
                     ])

# Preprocessing of training data, fit model 
clf.fit(X_train, y_train)

# Preprocessing of validation data, get predictions
preds = clf.predict(X_valid)

print('MAE:', mean_absolute_error(y_valid, preds))

MAE: 18017.665970319635


## Prediksi Data Test

Gunakan pipeline yang sudah dilatih untuk memprediksi `SalePrice` pada data test.
Pipeline otomatis menerapkan preprocessing yang sama seperti saat training — tidak perlu langkah tambahan.

In [6]:
preds_test = clf.predict(X_2)
print(preds_test)

[127436.5  159961.25 178998.25 ... 155623.87 106376.75 230180.9 ]


## Export Submission untuk Kaggle

Format submission Kaggle House Prices membutuhkan dua kolom:
- **`ID`** — ID rumah dari data test (diambil dari index `X_2`)
- **`SalePrice`** — hasil prediksi model

Buat DataFrame dari kedua kolom tersebut lalu export ke `Submission.csv` dengan `index=False`.

---

## Rangkuman

| Komponen | Fungsi |
|---|---|
| `SimpleImputer` | Mengisi missing value |
| `OneHotEncoder` | Mengubah data kategorik menjadi numerik |
| `ColumnTransformer` | Menerapkan transformer berbeda ke kolom berbeda |
| `Pipeline` | Menggabungkan semua langkah menjadi satu alur |
| `clf.fit()` | Melatih seluruh pipeline (preprocessing + model) |
| `clf.predict()` | Preprocessing + prediksi dalam satu pemanggilan |

---

## Tips

- Selalu gunakan pipeline saat ada preprocessing — mencegah *data leakage* antara training dan validation set.
- `fit_transform` hanya boleh dipanggil pada **training set**. Pipeline menangani ini secara otomatis.
- Pipeline dapat disimpan dan dimuat kembali untuk deployment:
  ```python
  import joblib
  joblib.dump(clf, 'model.pkl')   # simpan
  clf = joblib.load('model.pkl')  # muat kembali
  ```

In [8]:
output = pd.DataFrame({
    'ID' : X_2.index,
    'SalePrice': preds_test})

output.to_csv('Submission.csv', index=False)